# سبق 13 - ایجنٹ میموری


## ترتیب

یہ نوٹ بک ظاہر کرتی ہے کہ **Microsoft Agent Framework** (MAF) کا استعمال کرتے ہوئے **مسلسل یادداشت** کے ساتھ ایک سفری بکنگ ایجنٹ کیسے بنایا جائے۔

آپ سیکھیں گے کہ ایجنٹ کی مختلف قسم کی یادداشت — ورکنگ، قلیل مدتی، اور طویل مدتی — کس طرح ایک ایجنٹ کو بات چیت کے دوران معلومات کو برقرار رکھنے اور استعمال کرنے پر اثر انداز کرتی ہے۔

**ضروریات:**
- ایک Azure AI Foundry پروجیکٹ جس میں ایک تعینات چیٹ ماڈل ہو (مثلاً `gpt-4o-mini`)۔
- Azure CLI میں لاگ ان ہونا — اپنے ٹرمینل میں `az login` چلائیں۔
- `AZURE_AI_PROJECT_ENDPOINT` — آپ کے Azure AI Foundry پروجیکٹ کا اینڈ پوائنٹ۔
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — آپ کے تعینات کردہ ماڈل کا نام۔


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## ایجنٹ میموری کی اقسام

AI ایجنٹس مختلف اقسام کی میموری استعمال کر سکتے ہیں، ہر ایک کا ایک منفرد مقصد ہوتا ہے:

### ورکنگ میموری
گفتگو کا سلسلہ خود — جو پیغامات ایک ہی سیشن میں تبادلہ کیے جاتے ہیں۔ ایجنٹ اسی تھریڈ میں پہلے کے پیغامات کو دیکھ سکتا ہے تاکہ تسلسل برقرار رکھا جا سکے۔ MAF میں یہ **`agent.create_session()`** کے ذریعے بنایا جاتا ہے، جو ایک `AgentSession` واپس کرتا ہے۔

### قلیل مدتی میموری
وہ معلومات جو کسی کام یا سیشن کے دوران برقرار رہتی ہیں لیکن مستقل محفوظ نہیں ہوتیں۔ مثال کے طور پر، ایجنٹ متعدد مراحل پر محیط منصوبہ بندی کی گفتگو کے دوران حقائق جمع کر سکتا ہے اور ان کا استعمال حتمی روٹ بنانے کے لیے کر سکتا ہے۔

### طویل مدتی میموری
ترجیحات اور حقائق جو **سیشنز کے درمیان** برقرار رہتے ہیں۔ واپسی کرنے والے صارف کو اپنی خوراک کی پابندیاں یا سفر کا انداز بار بار دہرانا نہیں پڑنا چاہیے۔ طویل مدتی میموری عام طور پر ایک بیرونی ذخیرے — جیسے ڈیٹا بیس، فائل، یا ویکٹر انڈیکس — کے ذریعے سپورٹ کی جاتی ہے اور ایجنٹ کو ٹولز کے ذریعے مہیا کی جاتی ہے۔


## سیشنز کے ساتھ ورکنگ میموری

یادداشت کی سب سے سادہ شکل مکالمے کا سیشن ہے۔ جب آپ ایک ہی سیشن (جو `agent.create_session()` کے ذریعے بنایا گیا ہو) کو مسلسل `agent.run()` کالز میں پاس کرتے ہیں، تو ایجنٹ اس مکالمے کی مکمل تاریخ دیکھ سکتا ہے اور پہلے کے تفصیلات کو یاد رکھ سکتا ہے۔

آئیے ایک سفر ایجنٹ بنائیں اور ورکنگ میموری کا مظاہرہ کریں۔


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

ایجنٹ نے بجٹ کو صحیح طور پر یاد رکھا کیونکہ دونوں پیغامات ایک ہی سیشن شیئر کرتے ہیں۔ یہ **ورکنگ میموری** ہے — یہ صرف سیشن کی مدت کے لیے موجود ہے۔

### نئے تھریڈ کے ساتھ کیا ہوتا ہے؟

اگر ہم ایک **نیا** سیشن بنائیں، تو ایجنٹ کو پچھلی بات چیت کی کوئی یادداشت نہیں ہوتی:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## طویل مدتی یادداشت کا نمونہ

صارف کی ترجیحات کو **سیشنز کے دوران** یاد رکھنے کے لیے، ہمیں ایک مستقل ذخیرہ درکار ہے جو گفتگو کے دھاگے سے باہر موجود ہو۔ ایجنٹ اس ذخیرے تک رسائی حاصل کرتا ہے **آلات** کے ذریعے — وہ افعال جو وہ معلومات محفوظ کرنے اور بازیافت کرنے کے لیے کال کر سکتا ہے۔

نیچے ہم ایک سادہ ان-میموری ترجیحی ذخیرہ نافذ کرتے ہیں (پیداوار میں آپ اسے ایک ڈیٹا بیس یا ویکٹر انڈیکس کے ساتھ بیک کریں گے) اور اسے آلات کے طور پر ظاہر کرتے ہیں جنہیں ایجنٹ استعمال کر سکتا ہے۔

### فن تعمیر
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### منظر نامہ 1 — پہلی بار صارف سالگرہ کی سفر کی بکنگ کرتا ہے

سارہ پہلی بار آتی ہے۔ ایجنٹ کو چاہیے کہ وہ ٹولز کے ذریعے اس کی ترجیحات کو محفوظ کرے اور ہوٹلوں کی سفارش کے لیے ان کا استعمال کرے۔


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### منظر نامہ 2 — سارہ ہفتوں بعد واپس آتی ہے

سارہ ایک **نئی تھریڈ** شروع کرتی ہے (نئی سیشن کی نقل کرتے ہوئے)۔ ورکنگ میموری خالی ہے، لیکن طویل مدتی ترجیحات کا ذخیرہ اب بھی اس کی معلومات رکھتا ہے۔ ایجنٹ کو اسے بازیافت کرنا چاہیے اور ذاتی نوعیت کی سفارشات کے لیے اس کا استعمال کرنا چاہیے۔


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## خلاصہ

اس سبق میں آپ نے ایجنٹ کی تین اقسام کی میموری سیکھی اور انہیں Microsoft Agent Framework کے ساتھ کیسے نافذ کیا جائے:

| میموری کی قسم | MAF میکانزم | زندگی کی مدت |
|---|---|---|
| **ورکنگ** | `agent.create_session()` | ایک ہی گفتگو |
| **شارٹ ٹرم** | تھریڈ کے اندر جمع شدہ سیاق و سباق | ایک ہی کام / سیشن |
| **لانگ ٹرم** | ایکسٹرنل اسٹور جس تک `@tool` فنکشنز کے ذریعے رسائی حاصل ہوتی ہے | سیشنز کے دوران |

### اہم نکات
1. **`agent.create_session()`** ورکنگ میموری فراہم کرتا ہے — ایجنٹ سیشن کے اندر پوری گفتگو کی تاریخ دیکھتا ہے۔
2. **نئے سیشنز سیاق و سباق کھو دیتے ہیں** — بغیر لانگ ٹرم میموری کے ایجنٹ پچھلی گفتگو یاد نہیں رکھ سکتا۔
3. **`@tool` فنکشنز فرق کو ختم کرتے ہیں** — یہ فنکشنز ایجنٹ کو معلومات کو مستقل ذخیرے میں محفوظ اور بازیافت کرنے دیتے ہیں۔
4. **شخصی سازی وقت کے ساتھ بہتر ہوتی ہے** — جتنی زیادہ ترجیحات محفوظ ہوں گی، ایجنٹ کی سفارشات اتنی ہی بہتر ہوں گی۔

### حقیقی دنیا کی ایپلیکیشنز
- **کسٹمر سروس**: کسٹمر کی تاریخ اور ترجیحات کو یاد رکھنا
- **ذاتی معاونین**: دنوں یا ہفتوں کے درمیان سیاق و سباق برقرار رکھنا
- **صحت کی دیکھ بھال**: مریض کی معلومات اور ترجیحات کا پتہ لگانا
- **ای کامرس**: تاریخ کی بنیاد پر شخصی خریداری

### اگلے مراحل
- ان میموری ڈکشنری کو ڈیٹا بیس یا ویکٹر اسٹور (مثلاً Azure AI Search) سے تبدیل کریں
- وقت کے حساس معلومات کے لیے میموری کی میعاد مقرر کریں
- مشترکہ میموری کے ساتھ ملٹی ایجنٹ سسٹمز بنائیں
- Cognee نوٹ بک کو دریافت کریں جو علم گراف کی بنیاد پر میموری فراہم کرتی ہے


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**اخطار**:  
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کا استعمال کرتے ہوئے ترجمہ کی گئی ہے۔ اگرچہ ہم درستگی کی کوشش کرتے ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا ناکامیاں ہو سکتی ہیں۔ اصل دستاویز اپنی مادری زبان میں معتبر ماخذ سمجھا جانا چاہیے۔ اہم معلومات کے لیے پیشہ ورانہ انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے ہونے والی کسی بھی غلط فہمی یا غلط تعبیر کی ذمہ داری ہم پر عائد نہیں ہوتی۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
